# FreeFaceless - YouTube Shorts Generator
**Satu klik, langsung jalan.**

**Sebelum mulai, siapkan:**
1. Groq API Key (gratis): https://console.groq.com/keys  
2. Pexels API Key (gratis): https://www.pexels.com/api/

## 1. Isi API Key & Konfigurasi

In [ ]:
# ===== ISI API KEY DI SINI =====
GROQ_API_KEY = "gsk_...."  # ganti dengan key Groq-mu
PEXELS_API_KEY = "...."     # ganti dengan key Pexels-mu

# ===== KONFIGURASI =====
NICHE = "fakta sains unik yang jarang diketahui"
VOICE = "id-ID-ArdiNeural"
DURASI = 55  # detik

print("OK - siap lanjut")

## 2. Install Dependencies
*(Jalankan sekali. Kalo udah pernah, aman aja di-run ulang.)*

In [ ]:
print("Menginstall Python packages...")
!pip install -q openai edge-tts faster-whisper requests python-dotenv PyYAML
print("Menginstall ffmpeg...")
!apt-get -qq install ffmpeg
print("OK - semua dependencies siap")

## 3. Clone Project FreeFaceless

In [ ]:
import os, shutil
if os.path.exists("FreeFaceless"):
    shutil.rmtree("FreeFaceless")
!git clone -q https://github.com/nils44344/FreeFaceless.git
os.chdir("FreeFaceless")
print("FreeFaceless siap")

## 4. Setup Konfigurasi Indonesia + Progress Log

*(Semua file .py dimodifikasi biar keliatan progres real-time)*

In [ ]:
import os, re, time
from datetime import datetime
import yaml

# ----- .env -----
with open(".env", "w") as f:
    f.write(f"LLM_PROVIDER=groq\n")
    f.write(f"GROQ_API_KEY={GROQ_API_KEY}\n")
    f.write(f"PEXELS_API_KEY={PEXELS_API_KEY}\n")
print("[1/6] .env dibuat")

# ----- config.yaml -----
with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)
cfg["niche"] = NICHE
cfg["language"] = "id"
cfg["script"]["target_seconds"] = DURASI
cfg["voice"]["voice"] = VOICE
with open("config.yaml", "w") as f:
    yaml.dump(cfg, f)
print("[2/6] config.yaml diupdate")

# ----- script.py - Indonesian + progress -----
with open("src/script.py", "r") as f:
    code = f.read()

# Replace _system_prompt with Indonesian-only version
code = re.sub(
    r'def _system_prompt\(\):.*?(?=def generate)',
    '''def _system_prompt():
    s = CONFIG["script"]
    ts = s["target_seconds"]
    tw = int(ts * s["words_per_second"])
    return f"""Anda adalah penulis skrip YouTube Shorts viral untuk channel edukasi tanpa wajah (faceless).

Aturan:
- Skrip harus {ts} detik, ~{tw} kata total.
- Mulai dengan HOOK 1 kalimat yang bikin penasaran dalam <3 detik. Jangan pakai "Halo guys" atau perkenalan.
- Isi: 4-6 fakta mengejutkan yang akurat dan bisa diverifikasi.
- Akhiri dengan CTA 1 kalimat: "Ikuti untuk konten menarik lainnya".
- Gunakan bahasa Indonesia sehari-hari yang natural. Jangan pakai emoji atau format khusus.
- Setiap scene punya visual_query 2-4 kata benda bahasa Inggris untuk cari video stok di Pexels (misal: "octopus swimming ocean", "ancient pyramid desert").

Kembalikan ONLY valid JSON, tanpa teks lain. Skema:
{{"topic": "slug topik pendek", "title": "Judul YouTube max 95 chars, harus #shorts", "description": "2-3 kalimat deskripsi dengan hashtag", "tags": ["8-12 tag huruf kecil"], "scenes": [{{"text": "kalimat narasi bahasa Indonesia", "visual_query": "2-4 kata benda Inggris"}}]}}"""

"""
def generate():''',
    code, flags=re.DOTALL
)
code = code.replace('"""\ndef generate():', '')

# Force lang = "id" in generate()
code = re.sub(
    r'lang = CONFIG\.get\("language", "en"\)',
    'lang = "id"',
    code
)

# Add timing log
code = code.replace(
    'print(f"    calling {LLM_PROVIDER}/{LLM_MODEL}...")' if 'print(f"    calling' in code else '    resp = _call_llm(',
    't0 = time.time()\n    print(f"  calling LLM...", flush=True)\n    resp = _call_llm('
)
code = code.replace(
    'print(f"    LLM responded in {time.time()-t0:.1f}s ({len(raw)} chars)")' if 'print(f"    LLM responded' in code else '    raw = resp.choices[0].message.content',
    'print(f"  LLM responded in {time.time()-t0:.1f}s", flush=True)\n    raw = resp.choices[0].message.content'
)

with open("src/script.py", "w") as f:
    f.write(code)
print("[3/6] script.py - Indonesian + progress")

# ----- voice.py - progress -----
with open("src/voice.py", "r") as f:
    code = f.read()
code = code.replace(
    'import asyncio',
    'import asyncio, time'
)
code = code.replace(
    'async def _go():',
    't0 = time.time()\n    print(f"  voice: {len(text)} chars", flush=True)\n    async def _go():'
)
code = code.replace(
    'asyncio.run(_go())\n    return out_path',
    'asyncio.run(_go())\n    print(f"  voice done in {time.time()-t0:.1f}s", flush=True)\n    return out_path'
)
with open("src/voice.py", "w") as f:
    f.write(code)
print("[4/6] voice.py - progress")

# ----- visuals.py - per-scene progress + download % -----
with open("src/visuals.py", "r") as f:
    code = f.read()
code = code.replace(
    'import requests',
    'import time, requests'
)
code = code.replace(
    'def fetch_for_scenes(scenes: list[dict], out_dir: Path) -> list[Path]:\n    out_dir.mkdir(parents=True, exist_ok=True)\n    paths = []\n    for i, scene in enumerate(scenes):\n        url = search_vertical(scene["visual_query"])\n        if url is None:\n            url = search_vertical("abstract background")\n        if url is None:\n            raise RuntimeError(f"No Pexels result for scene {i}: {scene['visual_query']}")\n        paths.append(download(url, out_dir / f"scene_{i:02d}.mp4"))\n    return paths',
    '''def fetch_for_scenes(scenes: list[dict], out_dir: Path) -> list[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for i, scene in enumerate(scenes):
        q = scene["visual_query"]
        print(f"  scene {i+1}/{len(scenes)}: \"{q}\"", flush=True)
        t0 = time.time()
        url = search_vertical(q)
        if url is None:
            print(f"    no result, trying abstract background", flush=True)
            url = search_vertical("abstract background")
        if url is None:
            raise RuntimeError(f"No Pexels result for scene {i}: {q}")
        paths.append(download(url, out_dir / f"scene_{i:02d}.mp4"))
        print(f"    downloaded ({time.time()-t0:.0f}s)", flush=True)
    return paths'''
)
with open("src/visuals.py", "w") as f:
    f.write(code)
print("[5/6] visuals.py - progress")

# ----- pipeline.py - timestamp logs -----
with open("src/pipeline.py", "r") as f:
    code = f.read()
replacements = [
    ('print("[1/7] Generating script with Groq")',
     '_log("1/7 Generating script with LLM")'),
    ('print(f"      topic: {data[\'topic\']}")',
     '_log(f"    topic: {data[chr(39)+chr(116)+chr(111)+chr(112)+chr(105)+chr(99)]} ({len(data[chr(39)+chr(115)+chr(99)+chr(101)+chr(110)+chr(101)+chr(115)])} scenes)")'),
    ('print("[2/7] Synthesizing voiceover")',
     '_log("2/7 Synthesizing voiceover with Edge TTS")'),
    ('print("[3/7] Transcribing for word-level captions")',
     '_log("3/7 Transcribing for word-level captions (Faster-Whisper)")'),
    ('print("[4/7] Fetching b-roll from Pexels")',
     '_log("4/7 Fetching b-roll from Pexels")'),
    ('print("[5/7] Writing caption file")',
     '_log("5/7 Writing caption file")'),
    ('print("[6/7] Assembling final video with ffmpeg")',
     '_log("6/7 Assembling final video with ffmpeg")'),
    ('print(f"      output: {final}")',
     '_log(f"    final: {final.name} ({final.stat().st_size/1024/1024:.0f} MB)")'),
    ('print("[7/7] Uploading to YouTube")',
     '_log("7/7 Uploading to YouTube")'),
]
for old, new in replacements:
    code = code.replace(old, new)
with open("src/pipeline.py", "w") as f:
    f.write(code)
print("[6/6] pipeline.py - timestamps")

print("\n" + "=" * 50)
print("SEMUA SIAP. Pipeline bisa dijalankan.")
print("=" * 50)

## 5. JALANKAN PIPELINE

Progres akan keluar **real-time** di bawah ini. Paling lama di bagian Pexels download.

In [ ]:
import sys
sys.stdout.flush()
print("=" * 50)
print("MEMULAI PIPELINE...")
print("=" * 50)
print()
!python -u -m src.pipeline --no-upload 2>&1
print()
print("=" * 50)
print("SELESAI!")
print("=" * 50)

## 6. Download Hasil Video ke Google Drive

In [ ]:
from google.colab import drive
import glob, os

outputs = sorted(glob.glob("output/*/final.mp4"))
if outputs:
    latest = outputs[-1]
    folder = os.path.basename(os.path.dirname(latest))
    
    drive.mount("/content/drive")
    dest = f"/content/drive/MyDrive/FreeFaceless/{folder}"
    os.makedirs(dest, exist_ok=True)
    !cp -r "output/{folder}" "/content/drive/MyDrive/FreeFaceless/"
    mb = os.path.getsize(latest) / 1024 / 1024
    print(f"Video disimpan di: MyDrive/FreeFaceless/{folder}/final.mp4")
    print(f"Ukuran: {mb:.0f} MB")
else:
    print("Tidak ada output ditemukan. Pipeline mungkin gagal.")